---
title: "Practice Activity 7.1: Cross-Validation and Tuning"
author: Lily
format:
    html:
        embed-resources: true
        code-line-numbers: true
---


**GitHub Repository**: <https://github.com/lilysteinberg/GSB-544---Computing-and-ML/tree/main/Week%206>

In [38]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [39]:
ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


In [40]:
### data partitioning
#X_train, X_test, y_train, y_test = train_test_split(ames.drop(['SalePrice'], axis=1), ames[['SalePrice']], test_size=0.25)
X_train, X_test, y_train, y_test = train_test_split(ames.drop(['SalePrice'], axis=1), ames[['SalePrice']], random_state = 321)

# Part 1

In [41]:
# Pipeline for Model 1

#ames["TotRms AbvGrd"] = ames["TotRms AbvGrd"].astype(str)

m1_ct = ColumnTransformer(
    [
    ("standardize", StandardScaler(), ["Gr Liv Area"]),
    ("dummify", OneHotEncoder(sparse_output = False, handle_unknown="ignore"), ["TotRms AbvGrd"])
    ],
    remainder="drop"  # all other columns in X will be dropped.
)

m1_pipeline = Pipeline(
  [("preprocessing", m1_ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

m1_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area']),
                                                 ('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['TotRms AbvGrd'])])),
                ('linear_regression', LinearRegression())])

In [42]:
### strategies for checking the code works:
# 1. check to see if data is transformed as desired by ct
#m1_ct.fit_transform(X_train)

# 2. look at coefficients from regression
#m1_fitted = m1_pipeline.fit(X_train, y_train)
#m1_fitted.named_steps['linear_regression'].coef_

# 3. for simple models, plug in values manually.
## Look at first row of training data and manually compute coefficients*predictor values and compare to output of .predict()

# 4. visualization

In [43]:
# Pipeline for Model 2

m2_ct = ColumnTransformer(
    [
    ("standardize", StandardScaler(), ["Gr Liv Area"]),
    ("dummify", OneHotEncoder(sparse_output = False, handle_unknown="ignore"), ["TotRms AbvGrd", "Bldg Type"])
    ],
    remainder="drop"  # all other columns in X will be dropped.
)

m2_pipeline = Pipeline(
  [("preprocessing", m2_ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

m2_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area']),
                                                 ('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['TotRms AbvGrd',
                                                   'Bldg Type'])])),
                ('linear_regression', LinearRegression())])

In [44]:
# Pipeline for Model 3

m3_ct_dummies = ColumnTransformer(
    [
    ("standardize", StandardScaler(), ["Gr Liv Area"]),
    ("dummify", OneHotEncoder(sparse_output = False, handle_unknown="ignore"), ["Bldg Type"])
    ],
    remainder="passthrough" # all other columns in X are passed into the next step
)

m3_ct_interaction = ColumnTransformer(
    [
    ("interaction", PolynomialFeatures(interaction_only = True), ["dummify__Bldg Type_1Fam", "standardize__Gr Liv Area"])
    ],
    remainder="drop"  # all other columns in X will be dropped.
)

m3_pipeline = Pipeline(
  [("preprocessing1", m3_ct_dummies),
   ("preprocessing2", m3_ct_interaction),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

m3_pipeline

Pipeline(steps=[('preprocessing1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area']),
                                                 ('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Bldg Type'])])),
                ('preprocessing2',
                 ColumnTransformer(transformers=[('interaction',
                                                  PolynomialFeatures(interaction_only=True),
                                                  ['dummify__Bldg Type_1Fam',
                                                   'standardize__Gr Liv '
                                                   'Area'])])),
                ('linear_regression', LinearRegression())])

In [45]:
# checking how data flows thru the two different column transformations
X_train_dummified = m3_ct_dummies.fit_transform(X_train)
m3_ct_interaction.fit_transform(X_train_dummified)

,interaction__1,interaction__dummify__Bldg Type_1Fam,interaction__standardize__Gr Liv Area,interaction__dummify__Bldg Type_1Fam standardize__Gr Liv Area
853,1.0,1.0,-1.190548,-1.190548
1055,1.0,1.0,0.386700,0.386700
2483,1.0,1.0,-0.897883,-0.897883
2351,1.0,1.0,0.076358,0.076358
1700,1.0,1.0,2.445176,2.445176
...,...,...,...,...
1425,1.0,1.0,1.498435,1.498435
1833,1.0,1.0,2.669094,2.669094
2847,1.0,1.0,0.074394,0.074394
124,1.0,1.0,-0.901811,-0.901811


In [46]:
# Pipeline for Model 4

m4_ct = ColumnTransformer(
    [
    ("dummify", OneHotEncoder(sparse_output = False, handle_unknown="ignore"), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"]),
    ("polynomial_size", PolynomialFeatures(5, include_bias = False), ["TotRms AbvGrd"]),
    ("polynomial_rooms", PolynomialFeatures(5, include_bias = False), ["Gr Liv Area"])
    ],
    remainder="drop"
)

m4_pipeline = Pipeline(
  [("preprocessing", m4_ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

m4_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('dummify',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Bldg Type']),
                                                 ('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area',
                                                   'TotRms AbvGrd']),
                                                 ('polynomial_size',
                                                  PolynomialFeatures(degree=5,
                                                                     include_bias=False),
                                                  ['TotRms AbvGrd']),
                                                 ('polynomial_rooms',
                                                  PolynomialFeatures(degree=5,
                                                                     include_bias=False),
                                                  ['Gr Liv Area'])])),
                ('linear_regression', LinearRegression())])

**Model 1**

In [47]:
# model 1
m1_fitted = m1_pipeline.fit(X_train, y_train)

In [48]:
# model 1 - r2
y_preds = m1_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.47141603059965576

In [49]:
# model 1 - RMSE
root_mean_squared_error(y_test, y_preds)

53332.63518375074

**Model 2**

In [50]:
# model 2
m2_fitted = m2_pipeline.fit(X_train, y_train)

In [51]:
# model 2 - r2
y_preds = m2_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.4978252416259088

In [52]:
# model 2 - RMSE
root_mean_squared_error(y_test, y_preds)

51983.25722336787

**Model 3**

In [53]:
# model 3
m3_fitted = m3_pipeline.fit(X_train, y_train)

In [54]:
# model 3 - r2
y_preds = m3_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.4596474096662414

In [55]:
# model 3 - RMSE
root_mean_squared_error(y_test, y_preds)

53923.07718086932

**Model 4**

In [56]:
# model 4
m4_fitted = m4_pipeline.fit(X_train, y_train)

In [57]:
# model 4 - r2
y_preds = m4_fitted.predict(X_test)
r2_score(y_test, y_preds)

0.45338579528077927

In [58]:
# model 4 - RMSE
root_mean_squared_error(y_test, y_preds)

54234.60801107304

Model 2 performed the best.

# Part 2

In [59]:
# setting x and y variables for cross validation
X = ames.drop("SalePrice", axis = 1)
y = ames["SalePrice"]

**Model 1**

In [60]:
# model 1 - r2
m1_scores = cross_val_score(m1_pipeline, X, y, cv=5, scoring='r2')
m1_scores.mean()

np.float64(0.4963842325423311)

In [61]:
# model 1 - rmse
m1_scores_mse = cross_val_score(m1_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
m1_scores_mse.mean()

-m1_scores_mse.mean()
np.sqrt(-m1_scores_mse.mean())

np.float64(56453.06050628113)

**Model 2**

In [62]:
# model 2 - r2
m2_scores = cross_val_score(m2_pipeline, X, y, cv=5, scoring='r2')
m2_scores.mean()

np.float64(0.5268177513983843)

In [63]:
# model 2 - rmse
m2_scores_mse = cross_val_score(m2_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
m2_scores_mse.mean()

-m2_scores_mse.mean()
np.sqrt(-m2_scores_mse.mean())

np.float64(54675.1527201463)

**Model 3**

In [64]:
# model 3 - r2
m3_scores = cross_val_score(m3_pipeline, X, y, cv=5, scoring='r2')
m3_scores.mean()

np.float64(0.5028425816121013)

In [65]:
# model 3 - rmse
m3_scores_mse = cross_val_score(m3_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
m3_scores_mse.mean()

-m3_scores_mse.mean()
np.sqrt(-m3_scores_mse.mean())

np.float64(55966.12148322785)

**Model 4**

In [66]:
# model 4 - r2
m4_scores = cross_val_score(m4_pipeline, X, y, cv=5, scoring='r2')
m4_scores.mean()

np.float64(0.4971395761035386)

In [67]:
# model 4 - rmse
m4_scores_mse = cross_val_score(m1_pipeline, X, y, cv=5, scoring='neg_mean_squared_error')
m4_scores_mse.mean()

-m4_scores_mse.mean()
np.sqrt(-m4_scores_mse.mean())

np.float64(56453.06050628113)

Model 2 is still performing the best.

# Part 3

In [68]:
# set up grid search
m5_ct = ColumnTransformer(
    [
    ("dummify", OneHotEncoder(sparse_output = False, handle_unknown="ignore"), ["Bldg Type"]),
    ("standardize", StandardScaler(), ["Gr Liv Area", "TotRms AbvGrd"]),
    ("polynomial_size", PolynomialFeatures(), ["TotRms AbvGrd"]),
    ("polynomial_rooms", PolynomialFeatures(), ["Gr Liv Area"])
    ],
    remainder="drop"
)

m5_pipeline = Pipeline(
  [("preprocessing", m5_ct),
  ("linear_regression", LinearRegression())]
).set_output(transform="pandas")

m5_pipeline

degrees = {
    'preprocessing__polynomial_size__degree': np.arange(1, 11),
    'preprocessing__polynomial_rooms__degree': np.arange(1,11)
}

gscv = GridSearchCV(m5_pipeline, degrees, cv = 5, scoring='r2')

In [69]:
# feed grid search the data
gscv_fitted = gscv.fit(X, y)

gscv_fitted.cv_results_

{'mean_fit_time': array([0.04194813, 0.02862792, 0.03103437, 0.02741599, 0.03245177,
        0.02705092, 0.02809591, 0.04437127, 0.05609336, 0.03260198,
        0.06198416, 0.04738078, 0.0762156 , 0.0660955 , 0.05101686,
        0.03903232, 0.03192477, 0.02760878, 0.02992687, 0.03034444,
        0.03747535, 0.0296803 , 0.02945981, 0.03003879, 0.03805695,
        0.07551341, 0.04716544, 0.04765391, 0.04585319, 0.04397097,
        0.03885074, 0.06056623, 0.04488544, 0.06342425, 0.10014615,
        0.09087062, 0.09595623, 0.05159478, 0.04630136, 0.05722404,
        0.05596642, 0.10463624, 0.1388629 , 0.10115666, 0.06599793,
        0.04540873, 0.02867689, 0.03038192, 0.03442016, 0.02880054,
        0.02821193, 0.03052516, 0.03245363, 0.03032312, 0.03024688,
        0.03483825, 0.02963581, 0.03519101, 0.03082509, 0.03021069,
        0.02683992, 0.03386245, 0.03182721, 0.02782922, 0.02789426,
        0.03634577, 0.03046656, 0.03146467, 0.03074174, 0.03776627,
        0.03120494, 0.02890043,

In [70]:
# narrow down the results to view
gscv_fitted.cv_results_['mean_test_score']

array([ 0.53288244,  0.53238285,  0.53592417,  0.54152875,  0.54106618,
        0.53486226,  0.08006934, -1.09028711, -1.3014661 , -0.49485022,
        0.53747194,  0.53356735,  0.53413413,  0.5354176 ,  0.53026731,
        0.53331356,  0.35249861, -0.17703818, -0.2638761 ,  0.12995691,
        0.55764064,  0.55685725,  0.55403903,  0.55039244,  0.54654917,
        0.54525831,  0.55022195,  0.27580303,  0.02788513, -0.08442902,
        0.4992163 ,  0.50882755,  0.52832918,  0.52501528,  0.52119231,
        0.53129597,  0.51274932,  0.50139614,  0.524255  ,  0.44623187,
        0.49713958,  0.49713958,  0.49713958,  0.49713958,  0.49713958,
        0.50474232,  0.49393389,  0.48329217,  0.43985925,  0.11665322,
        0.44472675,  0.44472675,  0.44472675,  0.44472675,  0.44472675,
        0.44472675,  0.44472675,  0.44472675,  0.30756242,  0.26638373,
        0.26412025,  0.26412025,  0.26412025,  0.26412025,  0.26412025,
        0.26412025,  0.26412025,  0.26412025,  0.26412025,  0.26

In [71]:
# convert results into pandas df for easy viewing
scores = pd.DataFrame({
    "degrees_Size": np.repeat(np.arange(1, 11), 10),
    "degrees_Rooms": np.tile(np.arange(1, 11), 10),
    "scores": gscv_fitted.cv_results_['mean_test_score']
})

scores

,degrees_Size,degrees_Rooms,scores
0,1,1,0.532882
1,1,2,0.532383
2,1,3,0.535924
3,1,4,0.541529
4,1,5,0.541066
...,...,...,...
95,10,6,-6.286980
96,10,7,-6.286980
97,10,8,-6.286980
98,10,9,-6.286980


In [72]:
# make it a matrix for even easier viewing
scores_matrix = scores.pivot(index="degrees_Size", columns="degrees_Rooms", values="scores")
scores_matrix

degrees_Rooms,1,2,3,4,5,6,7,8,9,10
degrees_Size,,,,,,,,,,
1,0.532882,0.532383,0.535924,0.541529,0.541066,0.534862,0.080069,-1.090287,-1.301466,-0.494850
2,0.537472,0.533567,0.534134,0.535418,0.530267,0.533314,0.352499,-0.177038,-0.263876,0.129957
3,0.557641,0.556857,0.554039,0.550392,0.546549,0.545258,0.550222,0.275803,0.027885,-0.084429
4,0.499216,0.508828,0.528329,0.525015,0.521192,0.531296,0.512749,0.501396,0.524255,0.446232
5,0.497140,0.497140,0.497140,0.497140,0.497140,0.504742,0.493934,0.483292,0.439859,0.116653
6,0.444727,0.444727,0.444727,0.444727,0.444727,0.444727,0.444727,0.444727,0.307562,0.266384
7,0.264120,0.264120,0.264120,0.264120,0.264120,0.264120,0.264120,0.264120,0.264120,0.264120
8,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448,-0.269448
9,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519,-1.832519


In [73]:
# checking that I found the highest R-squared value in the matrix
gscv_fitted.cv_results_['mean_test_score'].max()

np.float64(0.5576406367909267)

Third degree for size and first degree for room number had the best results, as determined with the highest R-squared value.

A downside to trying all possible models is the time and computational resources required to run all of the combinations.

I might try running models in parallel, each separately raising one of the variables to the first, fourth, seventh, and tenth degrees. From those, I can see if the models are performing roughly better or worse at different points within the 1-10 number range. Once I have parsed down a smaller, more promising range of numbers (like 1-4), I would run the model with all combinations within that range and that range only. This would reduce the amount of models to run, but may increase some of the manual work on my end.